# 01. Exploratory Data Analysis & Schema Audit
> **Inspecting TMDB 5000, survivorship bias, and corrupted scale entries**

Part of the **GreenlightML** open-source film risk modeling and portfolio optimization engine.


## 1. Environment & Data Ingestion
Load required dependencies and inspect the raw TMDB 5000 CSV files.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))
from src.data import load_and_merge_data, filter_clean_theatrical

# Load unified dataset (deduplicating id=movie_id and duplicate titles)
df_raw = load_and_merge_data()
print(f"Raw unified dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} unique columns")
df_raw.head(3)


## 2. Audit of Missing Values and Zeros
Quantify unrecorded production budgets and theatrical box office receipts.


In [ ]:
zero_budget = (df_raw['budget'] == 0).sum()
zero_revenue = (df_raw['revenue'] == 0).sum()
zero_either = ((df_raw['budget'] == 0) | (df_raw['revenue'] == 0)).sum()

print(f"Zero Budget Entries:  {zero_budget} ({zero_budget / len(df_raw) * 100:.2f}%)")
print(f"Zero Revenue Entries: {zero_revenue} ({zero_revenue / len(df_raw) * 100:.2f}%)")
print(f"Total Incomplete:     {zero_either} ({zero_either / len(df_raw) * 100:.2f}%)")

df_clean = filter_clean_theatrical(df_raw)
print(f"Clean Theatrical Sample: {len(df_clean)} titles remain.")


## 3. Detecting Currency Scaling Corruptions
Audit micro-budget records (<$100,000) for missing million-scale multipliers.


In [ ]:
# Detect entries with budgets under $1,000 entered without multiplier
micro_budgets = df_clean[df_clean['budget'] < 1000][['title', 'budget', 'revenue', 'release_date']]
print(f"Detected {len(micro_budgets)} micro-budget records:")
micro_budgets.head(10)


## 4. The Survivorship Trap & Class Distribution
Demonstrate the 75.5% majority class imbalance on commercially reported titles.


In [ ]:
target_dist = df_clean['target'].value_counts(normalize=True) * 100
print(f"Commercial Success (Revenue > Budget): {target_dist.get(1, 0):.2f}%")
print(f"Theatrical Flop    (Revenue <= Budget): {target_dist.get(0, 0):.2f}%")

plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x='target', palette=['#ef4444', '#10b981'])
plt.title("Theatrical Success vs. Flop Distribution (Survivorship Bias)")
plt.xticks([0, 1], ['Flop (24.5%)', 'Success (75.5%)'])
plt.ylabel("Number of Titles")
plt.show()


## 5. Correlation Analysis & Preliminary Leakage Audit
Examine correlations between numeric features and final box office gross.


In [ ]:
num_cols = ['budget', 'popularity', 'vote_average', 'vote_count', 'revenue']
corr_matrix = df_clean[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f')
plt.title("Correlation Matrix (Highlighting Post-Release Leakage Features)")
plt.show()
